In [1]:

import os
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report
from torch.optim import AdamW
from tqdm import tqdm
from ast import literal_eval

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

d:\System\Conda\envs\absa\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [2]:
def load_semeval(dataset, num_classes, data_path=None):
    """ Loads SemEval 14 datasets

    Arguments:
        dataset (str): Name of the datsets
        num_classes (int): Number of classes to consider
        data_path (str): Path to the dataset

    """
    label = {'negative': 0,
             'positive': 1,
             'neutral': 2,
             'conflict': 3}

    if(data_path is None):
        data_path = '../Data/'
    train_file = data_path + 'atsa-' + dataset + '/atsa_train.json'
    test_file = data_path + 'atsa-' + dataset + '/atsa_test.json'

    temp = open(train_file, 'r', encoding='latin1').read()
    train = literal_eval(temp)
    train_sentence = []
    train_aspect = []
    train_sentiment = []
    for xml in train:
        if(xml['sentiment'] == 'conflict' and num_classes == 3):
            continue
        train_sentence.append(xml['sentence'])
        train_aspect.append(xml['aspect'])
        train_sentiment.append(label[xml['sentiment']])

    temp = open(test_file, 'r', encoding='latin1').read()
    test = literal_eval(temp)
    test_sentence = []
    test_aspect = []
    test_sentiment = []
    for xml in test:
        if(xml['sentiment'] == 'conflict' and num_classes == 3):
            continue
        test_sentence.append(xml['sentence'])
        test_aspect.append(xml['aspect'])
        test_sentiment.append(label[xml['sentiment']])

    train_sen_len = [len(sentence.split()) for sentence in train_sentence]
    train_asp_len = [len(aspect.split()) for aspect in train_aspect]

    print("----------------------------------------")
    print("Maximum Training data Sentence Length: {}".format(
                                                       max(train_sen_len)))
    print("Maximum Training data Aspect Length: {}".format(max(train_asp_len)))
    print("----------------------------------------")

    return (train_sentence, train_aspect, train_sentiment, test_sentence,
            test_aspect, test_sentiment)

In [3]:
class ABSADataset(Dataset):
    def __init__(self, sentences, aspects, labels, tokenizer, max_len):
        self.sentences = sentences
        self.aspects = aspects
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.sentences[idx],
            self.aspects[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        item = {k: v.squeeze(0) for k, v in encoded.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [4]:
def train_kfold_model(model_name,
                      train_sentences, train_aspects, train_labels,
                      test_sentences, test_aspects, test_labels,
                      num_classes=3, k=5, epochs=3, batch_size=16):

    save_path = f"{model_name.replace('/', '_')}_semeval_model.pth"

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # If already trained → load and evaluate on test
    if os.path.exists(save_path):
        print(f"\nModel {model_name} already exists. Loading...")
        model = load_model(model_name, num_classes, save_path)
        evaluate_on_test(model, tokenizer,
                         test_sentences, test_aspects, test_labels)
        return model

    print(f"\nTraining {model_name} with {k}-Fold CV")

    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

    best_model_state = None
    best_f1 = 0
    fold_metrics = []

    for fold, (train_idx, val_idx) in enumerate(
            skf.split(train_sentences, train_labels)):

        print(f"\n========== Fold {fold+1} ==========")

        tr_sent = [train_sentences[i] for i in train_idx]
        tr_asp = [train_aspects[i] for i in train_idx]
        tr_lab = [train_labels[i] for i in train_idx]

        val_sent = [train_sentences[i] for i in val_idx]
        val_asp = [train_aspects[i] for i in val_idx]
        val_lab = [train_labels[i] for i in val_idx]

        train_dataset = ABSADataset(tr_sent, tr_asp, tr_lab, tokenizer, 128)
        val_dataset = ABSADataset(val_sent, val_asp, val_lab, tokenizer, 128)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size)

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_classes
        ).to(DEVICE)

        optimizer = AdamW(model.parameters(), lr=2e-5)

        # Training
        for epoch in range(epochs):
            model.train()
            total_loss = 0

            for batch in tqdm(train_loader):
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                outputs = model(**batch)
                loss = outputs.loss

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

        # Validation
        model.eval()
        preds, true = [], []

        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                outputs = model(**batch)
                logits = outputs.logits
                predictions = torch.argmax(logits, dim=1)

                preds.extend(predictions.cpu().numpy())
                true.extend(batch["labels"].cpu().numpy())

        acc = accuracy_score(true, preds)
        f1 = f1_score(true, preds, average="macro")

        print(f"Fold {fold+1} Accuracy: {acc:.4f}")
        print(f"Fold {fold+1} Macro-F1: {f1:.4f}")

        fold_metrics.append((acc, f1))

        if f1 > best_f1:
            best_f1 = f1
            best_model_state = model.state_dict()

    print("\n========== K-FOLD AVERAGE ==========")
    print("Avg Accuracy:", np.mean([m[0] for m in fold_metrics]))
    print("Avg Macro-F1:", np.mean([m[1] for m in fold_metrics]))

    # Save best model
    torch.save(best_model_state, save_path)
    print("Saved model at:", save_path)

    model.load_state_dict(best_model_state)

    # Final test evaluation
    evaluate_on_test(model, tokenizer,
                     test_sentences, test_aspects, test_labels)

    return model

In [5]:
def evaluate_on_test(model, tokenizer,
                     test_sentences, test_aspects, test_labels):

    test_dataset = ABSADataset(test_sentences, test_aspects,
                               test_labels, tokenizer, 128)
    test_loader = DataLoader(test_dataset, batch_size=16)

    model.eval()
    preds, true = [], []

    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=1)

            preds.extend(predictions.cpu().numpy())
            true.extend(batch["labels"].cpu().numpy())

    acc = accuracy_score(true, preds)
    f1 = f1_score(true, preds, average="macro")

    print("\n========== TEST SET RESULTS ==========")
    print("Test Accuracy:", acc)
    print("Test Macro-F1:", f1)
    print("\nClassification Report:\n")
    print(classification_report(true, preds))

In [6]:
def load_model(model_name, num_classes, path):
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_classes
    ).to(DEVICE)

    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    return model

In [7]:
# Load dataset
(train_sentence, train_aspect, train_sentiment,
 test_sentence, test_aspect, test_sentiment) = load_semeval(
    dataset="laptop",
    num_classes=3,
    data_path="../Data/"
)

models_to_train = [
    "bert-base-uncased",
    "roberta-base",
    "microsoft/deberta-base"
]

trained_models = {}

for model_name in models_to_train:
    model = train_kfold_model(
        model_name,
        train_sentence, train_aspect, train_sentiment,
        test_sentence, test_aspect, test_sentiment,
        num_classes=3,
        k=5,
        epochs=3,
        batch_size=16
    )
    trained_models[model_name] = model

----------------------------------------
Maximum Training data Sentence Length: 78
Maximum Training data Aspect Length: 6
----------------------------------------

Model bert-base-uncased already exists. Loading...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



========== TEST SET RESULTS ==========
Test Accuracy: 0.7586206896551724
Test Macro-F1: 0.7060792020092593

Classification Report:

              precision    recall  f1-score   support

           0       0.58      0.88      0.70       128
           1       0.87      0.87      0.87       341
           2       0.72      0.44      0.55       169

    accuracy                           0.76       638
   macro avg       0.72      0.73      0.71       638
weighted avg       0.77      0.76      0.75       638


Model roberta-base already exists. Loading...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



========== TEST SET RESULTS ==========
Test Accuracy: 0.8103448275862069
Test Macro-F1: 0.7769810341754386

Classification Report:

              precision    recall  f1-score   support

           0       0.64      0.89      0.75       128
           1       0.93      0.87      0.90       341
           2       0.74      0.63      0.68       169

    accuracy                           0.81       638
   macro avg       0.77      0.80      0.78       638
weighted avg       0.83      0.81      0.81       638


Model microsoft/deberta-base already exists. Loading...


Some weights of DebertaForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



========== TEST SET RESULTS ==========
Test Accuracy: 0.799373040752351
Test Macro-F1: 0.7608087912986291

Classification Report:

              precision    recall  f1-score   support

           0       0.64      0.86      0.74       128
           1       0.90      0.88      0.89       341
           2       0.74      0.59      0.65       169

    accuracy                           0.80       638
   macro avg       0.76      0.78      0.76       638
weighted avg       0.81      0.80      0.80       638



In [8]:
def predict(model, tokenizer, sentence, aspect):
    model.eval()

    encoded = tokenizer(
        sentence,
        aspect,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)
        probs = torch.softmax(outputs.logits, dim=1)
        confidence, pred = torch.max(probs, dim=1)

    return pred.item(), confidence.item()

In [10]:
label_map = {0: "Negative", 1: "Positive", 2: "Neutral"}

test_sent = "The battery life is bad but the screen is not bad."
test_asp = "battery life"

for model_name in models_to_train:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = trained_models[model_name]

    pred, conf = predict(model, tokenizer, test_sent, test_asp)
    print(f"\nModel: {model_name}")
    print("Prediction:", label_map[pred])
    print("Confidence:", f"{conf:.2%}")


Model: bert-base-uncased
Prediction: Negative
Confidence: 98.25%

Model: roberta-base
Prediction: Negative
Confidence: 95.80%

Model: microsoft/deberta-base
Prediction: Negative
Confidence: 68.79%
